# Phase 5 improvements — effectiveness audit on Phase 7 prep data

Uses the 8-study Phase 7 prep run (2026-04-20, post-Commit-G) as production-sized evidence for the Phase 5 signal-quality stack. This is not an A/B trial — there's no same-hull counterfactual-without-5D run at this scale — but each transform's *claimed mechanism* is measurable on the logged data.

**What's evaluated here:**

| Phase | Change | Measurable signature |
|---|---|---|
| **5A** TWFE | opponent-FE decomposition | Var(α̂) vs Var(raw_fitness); opponent-FE magnitudes per hull |
| **5B** ASHA + WilcoxonPruner | rung-based early stop | matchups saved; rung-at-prune distribution; prune-rate rank correlation |
| **5C** Opponent curriculum | anchor-first + incumbent-overlap | opponent-pull frequency spread; leading-opponent share |
| **5D** EB shrinkage | regression-prior fusion | shrinkage weight distribution; γ̂ covariate importance; τ̂² vs σ̂² ratios; |Δ| = |α̂ − α̂_EB| |
| **5E** Box-Cox + min-max | output warping | ceiling saturation (frac at fitness ≥ 0.99); tail compression; passthrough rate |
| **5F** Regime segmentation | N/A | only `early` regime present in this run — can't compare cross-regime |

In [ ]:
import json, glob, os, sys, warnings
from collections import Counter, defaultdict
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr, kendalltau

sys.path.insert(0, '../src')
from starsector_optimizer.deconfounding import eb_shrinkage
from starsector_optimizer.models import EBShrinkageConfig
warnings.filterwarnings('ignore', category=UserWarning)

LOG_GLOB = '../data/logs/*__early__tpe__seed0'
studies = {}
for d in sorted(glob.glob(LOG_GLOB)):
    hull = os.path.basename(d).split('__')[0]
    studies[hull] = [json.loads(l) for l in open(f'{d}/evaluation_log.jsonl') if l.strip()]
print(f'Loaded {len(studies)} studies:', {h: len(r) for h, r in studies.items()})

## 5A — TWFE decomposition

Claim: subtracting an opponent fixed-effect yields an α̂ that is less confounded by opponent-mix than raw mean Y_ij. Test: does Var(α̂) < Var(raw_fitness) across trials of a given hull? (If yes, the opponent-FE captured real variance, not just shuffled it.)

In [ ]:
twfe_rows = []
for hull, rows in studies.items():
    fin = [r for r in rows if not r.get('pruned') and r.get('twfe_fitness') is not None]
    if len(fin) < 30: continue
    raw  = np.array([r['raw_fitness']  for r in fin])
    twfe = np.array([r['twfe_fitness'] for r in fin])
    rho, _ = spearmanr(raw, twfe)
    twfe_rows.append({
        'hull': hull, 'n': len(fin),
        'SD(raw)':   round(raw.std(ddof=1), 4),
        'SD(twfe)':  round(twfe.std(ddof=1), 4),
        'var_reduction_pct': round(100 * (1 - twfe.var() / max(raw.var(), 1e-12)), 1),
        'ρ(raw, α̂)': round(rho, 3),
        'mean(raw)':  round(raw.mean(), 3),
        'mean(α̂)':    round(twfe.mean(), 3),
    })
pd.DataFrame(twfe_rows).set_index('hull')

Interpretation:
- **var_reduction_pct > 0**: TWFE explained part of the variance as opponent-FE — real signal separation. Negative means α̂ is noisier than raw (e.g., low-signal hulls where the FE estimation itself introduces noise).
- **ρ(raw, α̂) high (>0.8)**: rank-preserving transform; TWFE didn't *invert* the ordering, just re-weighted it.
- The hulls where TWFE buys little (frigates wolf/lasher) are exactly the ones Phase 5D was designed to help with via the EB prior — low-signal regimes where α̂ alone is too noisy.

## 5B — ASHA + WilcoxonPruner

Claims:
1. Early-rung pruning saves compute without materially harming top-k identification.
2. Pruning distribution should be rung-1-heavy (clear losers die fast); few prunes at later rungs (uncertain builds run longer).

In [ ]:
N_OPPS = 10  # rungs per trial
asha_rows = []
prune_rung_hist = defaultdict(Counter)
for hull, rows in studies.items():
    pruned_at = [r.get('opponents_evaluated', 0) for r in rows if r.get('pruned')]
    finalized = [r for r in rows if not r.get('pruned')]
    total = len(rows)
    saved_matchups  = sum(N_OPPS - r for r in pruned_at)
    used_matchups   = sum((r.get('opponents_evaluated') or 0) for r in rows)
    total_no_prune  = total * N_OPPS
    for r in pruned_at:
        prune_rung_hist[hull][r] += 1
    asha_rows.append({
        'hull': hull, 'total_trials': total,
        'pruned': len(pruned_at),
        'prune_rate': round(len(pruned_at)/max(total,1), 3),
        'mean_prune_rung': round(np.mean(pruned_at), 2) if pruned_at else None,
        'matchups_used':  used_matchups,
        'matchups_saved': saved_matchups,
        'compute_saved_pct': round(100 * saved_matchups / max(total_no_prune, 1), 1),
    })
pd.DataFrame(asha_rows).set_index('hull')

In [ ]:
# Rung-at-prune histogram — expect heavy tail at rung 1 (clear losers)
df_hist = pd.DataFrame(prune_rung_hist).fillna(0).astype(int)
df_hist = df_hist.reindex(sorted(df_hist.index))
df_hist.index.name = 'rung'
df_hist

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for hull in df_hist.columns:
    ax.plot(df_hist.index, df_hist[hull], 'o-', label=hull, alpha=0.7)
ax.set(xlabel='rung at prune', ylabel='# trials pruned', title='ASHA prune-rung distribution per hull')
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5C — Opponent curriculum

Phase 5C design: early rungs hit a set of *anchor* opponents (fixed for variance control); later rungs select by *incumbent overlap* (opponents the current-best struggled against). Signature: a handful of high-frequency anchors + a long tail of rarer opponents.

In [ ]:
opp_hist = {}
for hull, rows in studies.items():
    c = Counter()
    for r in rows:
        for opp_id in r.get('opponent_order', []) or []:
            c[opp_id] += 1
    opp_hist[hull] = c

# Top-5 anchors per hull and their total-pull share
anchor_rows = []
for hull, c in opp_hist.items():
    total = sum(c.values())
    if total == 0: continue
    top5 = c.most_common(5)
    anchor_rows.append({
        'hull': hull,
        'unique_opponents': len(c),
        'top1_share': round(top5[0][1] / total, 3),
        'top5_share': round(sum(n for _, n in top5) / total, 3),
        'top5_ids':   ', '.join(f'{k}×{v}' for k, v in top5),
    })
pd.DataFrame(anchor_rows).set_index('hull')

Interpretation: `top5_share` ≫ 0.5 with stable identity across trials signals effective anchor reuse. `top1_share` near 1/N_opps (10% for a uniform 10-opp pool) signals no curriculum — pure uniform sampling.

## 5D — Empirical-Bayes shrinkage

Claims:
1. The prior pulls raw α̂ toward γ̂ᵀ[1, X], reducing MSE on low-signal builds.
2. Shrinkage weight w_i = τ̂² / (τ̂² + σ̂_i²) scales naturally with per-build uncertainty.
3. γ̂ coefficients should be non-zero on admissible covariates, indicating the prior carries information.

In [ ]:
def refit_eb(rows):
    keep = [r for r in rows if r.get('twfe_fitness') is not None and r.get('covariate_vector') and not r.get('pruned')]
    if len(keep) < 8: return None, []
    alpha = np.array([r['twfe_fitness'] for r in keep], float)
    X = np.array([r['covariate_vector'] for r in keep], float)
    def s2(r):
        d = r.get('eb_diagnostics') or {}
        return d.get('sigma_sq_twfe') or float(np.var(alpha) / max(len(alpha)-1, 1))
    sigma2 = np.array([s2(r) for r in keep], float)
    eb, gamma, tau2, kept = eb_shrinkage(alpha, sigma2, X, EBShrinkageConfig())
    return {'alpha': alpha, 'eb': eb, 'gamma': gamma, 'tau2': tau2, 'kept': kept, 'sigma2': sigma2}, keep

ebfit = {h: refit_eb(rows)[0] for h, rows in studies.items()}

eb_summary = []
for hull, f in ebfit.items():
    if f is None: continue
    delta = f['eb'] - f['alpha']
    w = f['tau2'] / (f['tau2'] + f['sigma2'])
    eb_summary.append({
        'hull': hull, 'n': len(f['alpha']),
        'τ̂²': round(f['tau2'], 5),
        'median σ̂_i²': round(float(np.median(f['sigma2'])), 5),
        'median w_i': round(float(np.median(w)), 3),
        'mean |Δ|': round(float(np.mean(np.abs(delta))), 3),
        'max |Δ|':  round(float(np.max(np.abs(delta))), 3),
        'kept_cov_columns': f'{len(f["kept"])}/10',
    })
pd.DataFrame(eb_summary).set_index('hull')

In [ ]:
# EB correction scatter: α̂ vs α̂_EB per hull — points should cluster near y=x with slight pull toward the hull's global prior line.
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, (hull, f) in zip(axes.ravel(), ebfit.items()):
    if f is None: continue
    ax.scatter(f['alpha'], f['eb'], s=8, alpha=0.4)
    lo = min(f['alpha'].min(), f['eb'].min()); hi = max(f['alpha'].max(), f['eb'].max())
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.5, lw=0.8)
    ax.set_title(f'{hull}  τ̂²={f["tau2"]:.3g}'); ax.grid(alpha=0.3)
    ax.set_xlabel('α̂_TWFE'); ax.set_ylabel('α̂_EB')
plt.suptitle('EB shrinkage: α̂_TWFE → α̂_EB  (y=x dashed)', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# γ̂ covariate importance per hull
COV_NAMES = [
    'eff_max_flux', 'eff_flux_dissipation', 'eff_armor_rating',
    'eff_hull_hp_pct', 'ballistic_range_bonus', 'shield_damage_taken_mult',
    'total_weapon_dps', 'engagement_range', 'kinetic_dps_fraction',
    'op_used_fraction',
]

gamma_table = {}
for hull, f in ebfit.items():
    if f is None: continue
    row = {name: 0.0 for name in COV_NAMES}
    for i, col in enumerate(f['kept']):
        row[COV_NAMES[col]] = f['gamma'][1 + i]
    gamma_table[hull] = row
df_gamma = pd.DataFrame(gamma_table).T
df_gamma.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(df_gamma.values, cmap='RdBu_r', aspect='auto',
               vmin=-df_gamma.abs().max().max(), vmax=df_gamma.abs().max().max())
ax.set_xticks(range(len(COV_NAMES))); ax.set_xticklabels(COV_NAMES, rotation=45, ha='right')
ax.set_yticks(range(len(df_gamma))); ax.set_yticklabels(df_gamma.index)
for i in range(len(df_gamma)):
    for j in range(len(COV_NAMES)):
        v = df_gamma.values[i, j]
        if abs(v) > 0.01:
            ax.text(j, i, f'{v:+.2f}', ha='center', va='center', fontsize=7,
                    color='white' if abs(v) > 0.3 else 'black')
plt.colorbar(im, ax=ax, label='γ̂')
ax.set_title('EB prior γ̂ coefficients per hull × covariate')
plt.tight_layout(); plt.show()

## 5E — Box-Cox + min-max output warping

Claim: Box-Cox fixes TPE's ceiling-saturation problem (many raw fitnesses pile near the max after a few strong builds are found, collapsing top-k identification). Signature:
1. **Saturation**: fraction of `fitness` values ≥ 0.99 should be low (<5%) — target is 0.5% per plan.
2. **Lambda distribution**: λ far from 1 indicates non-trivial warping happening.
3. **Passthrough rate**: how often the < 8-sample floor is hit (EB insufficient → no warping).

In [ ]:
shape_rows = []
for hull, rows in studies.items():
    final = [r for r in rows if not r.get('pruned') and r.get('fitness') is not None]
    if not final: continue
    fits = np.array([r['fitness'] for r in final])
    lambdas = [r['shape_lambda'] for r in final if r.get('shape_lambda') is not None]
    passthrough = sum(1 for r in final if r.get('shape_passthrough_reason') and r['shape_passthrough_reason'] != 'ok')
    sat_099 = float((fits >= 0.99).mean())
    shape_rows.append({
        'hull': hull, 'n': len(final),
        'saturation@0.99': round(sat_099, 4),
        'median_λ': round(np.median(lambdas), 3) if lambdas else None,
        'IQR_λ': f'[{np.quantile(lambdas, 0.25):.2f}, {np.quantile(lambdas, 0.75):.2f}]' if lambdas else None,
        'passthrough_rate': round(passthrough / len(final), 3),
        'n_with_lambda': len(lambdas),
    })
pd.DataFrame(shape_rows).set_index('hull')

In [ ]:
# Passthrough reasons breakdown
pt_reasons = Counter()
for hull, rows in studies.items():
    for r in rows:
        reason = r.get('shape_passthrough_reason')
        if reason: pt_reasons[reason] += 1
pd.Series(dict(pt_reasons)).sort_values(ascending=False).head(10)

In [ ]:
# Compare fitness distribution vs eb_fitness distribution per hull — visualize tail compression
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, (hull, rows) in zip(axes.ravel(), studies.items()):
    final = [r for r in rows if not r.get('pruned') and r.get('eb_fitness') is not None and r.get('fitness') is not None]
    if not final: continue
    eb = np.array([r['eb_fitness'] for r in final])
    sh = np.array([r['fitness'] for r in final])
    ax.hist(eb, bins=30, alpha=0.5, label='EB α̂')
    ax.hist(sh, bins=30, alpha=0.5, label='shaped (Box-Cox+MM)')
    ax.axvline(0.99, color='red', ls='--', alpha=0.5, lw=0.8)
    ax.set_title(hull); ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8, loc='upper left')
plt.suptitle('EB vs shaped fitness distributions — red line = 0.99 saturation threshold', y=1.02)
plt.tight_layout(); plt.show()

## 5F — Regime segmentation

Not evaluable on this run — only `early` regime is present. Cross-regime comparison requires one `(hull, regime)` study pair with warm-start enabled. Scheduled for a dedicated sweep after Phase 7 prep completes.

## Roll-up: headline effectiveness numbers

One-line summaries that can go into the retrospective.

In [ ]:
import statistics
headline = {}

# 5A: mean variance reduction across hulls
vr = [r['var_reduction_pct'] for r in twfe_rows if r['var_reduction_pct'] is not None]
headline['5A mean var_reduction %']      = round(statistics.mean(vr), 1) if vr else None
headline['5A median ρ(raw, α̂)']         = round(statistics.median(r['ρ(raw, α̂)'] for r in twfe_rows), 3) if twfe_rows else None

# 5B: mean compute saved by ASHA
cs = [r['compute_saved_pct'] for r in asha_rows]
headline['5B mean ASHA compute saved %'] = round(statistics.mean(cs), 1) if cs else None
headline['5B mean prune rate']           = round(statistics.mean(r['prune_rate'] for r in asha_rows), 3) if asha_rows else None

# 5C: mean top-5 anchor share
headline['5C mean top-5 opponent share'] = round(statistics.mean(r['top5_share'] for r in anchor_rows), 3) if anchor_rows else None

# 5D: median shrinkage weight + mean |Δ|
headline['5D median EB weight']          = round(statistics.median(r['median w_i'] for r in eb_summary), 3) if eb_summary else None
headline['5D mean |Δα̂|']                = round(statistics.mean(r['mean |Δ|']   for r in eb_summary), 3) if eb_summary else None

# 5E: mean ceiling saturation
headline['5E mean saturation @ 0.99']    = round(statistics.mean(r['saturation@0.99'] for r in shape_rows), 4) if shape_rows else None

pd.Series(headline).to_frame('value')